# PhysicsNeMo 기초 개념

양끝이 고정된 줄의 처짐을 예측하려면 무엇을 알아야 할까요? 줄 전체에 적용되는 방정식과 양끝의 높이를 알면 해를 구할 수 있습니다. PINN은 이 두 조건을 이용해 신경망을 학습합니다.


[실습 목차](00_Start_Here.ipynb) · [PINN 실습](01_PINN.ipynb)


## 방정식으로 신경망 학습하기

신경망에 위치 $x$를 넣으면 그 지점의 변위 $u(x)$가 나옵니다. 예측 곡선을 미분해 방정식의 양변이 얼마나 어긋나는지 구하고, 양 끝점에서는 예측한 높이를 주어진 값과 비교합니다. 이 오차들을 모아 하나의 손실로 만든 뒤 신경망을 학습합니다.


## 실습에서 쓰는 도구

PhysicsNeMo는 PyTorch에서 쓸 수 있는 신경망과 물리 방정식 계산 도구를 제공합니다. 이 교재는 버전 2.2.2를 사용합니다.

[공식 문서](https://docs.nvidia.com/physicsnemo/latest/index.html) · [버전 2.2.2 소스](https://github.com/NVIDIA/physicsnemo/tree/v2.2.2)


| 도구 | 줄의 처짐 문제에서 하는 일 |
| --- | --- |
| `physicsnemo.models.mlp.FullyConnected` | 위치에서 변위를 예측하는 신경망을 구성합니다. |
| SymPy, `physicsnemo.sym.eq.pde.PDE` | 물리량과 미분 항을 식으로 정의합니다. |
| `PhysicsInformer` | 신경망이 예측한 변위로 방정식의 잔차를 계산합니다. |
| PyTorch 옵티마이저 | 내부 지점의 잔차와 양 끝점의 오차를 줄이도록 신경망의 가중치를 갱신합니다. |


교재의 [학습 보조 함수](ETC/runtime/labs.py)에서 `mlp`는 PhysicsNeMo 신경망을 만들고, `informer`는 식을 받아 잔차 계산기를 설정합니다. 줄의 문제에서 이 함수를 어떻게 호출하는지는 [PINN 실습 코드](01_labs/01_pinn/source_code/pinn_basics.py)에서 확인할 수 있습니다.


### 학습 코드 따라가기

1. `Poisson1D`에서 줄의 방정식을 정의하고, `BasicPINN`에서 신경망을 만듭니다.
2. `loss_terms`에서 신경망의 예측값을 방정식에 넣고 양끝의 변위를 계산합니다. 역문제에서는 관측한 높이와의 차이도 손실에 더합니다.
3. `optimize_lab`에서 L-BFGS로 가중치를 갱신합니다. 학습이 끝나면 401개 위치에서 변위를 해석해와 비교합니다.


## PINN, FNO, PINO

줄의 PINN은 위치를 입력으로 받고 방정식과 경계조건으로 학습합니다. 반면 [열판의 FNO](02_FNO.ipynb)는 판 전체의 발열 밀도 지도를 받아 온도 지도를 예측합니다. 여러 히터 조건에서 계산한 입력·출력 지도 쌍이 학습 데이터가 됩니다. 신경 연산자를 학습할 때 방정식 잔차도 손실에 넣는 방식을 PINO라고 합니다.

[FourCastNet 날씨 실습](01_labs/04_weather_forecasting/Lab_4_Weather_Forecasting.ipynb)에서는 이미 학습된 AFNO 모델을 사용합니다. 전 지구의 26개 기상 변수로 6시간 뒤 상태를 예측하고, 이를 다음 입력으로 넣어 48시간 예보를 만듭니다. 이후 시각의 ERA5 재분석 자료와 비교해 오차를 구합니다.


### 무엇을 미분할까요?

```python
from physicsnemo.models.mlp import FullyConnected

model = FullyConnected(in_features=1, out_features=1, layer_size=64,
                       num_layers=3, activation_fn="tanh")
```

이 신경망이 위치 $x$에서 변위 $u_\theta(x)$를 예측한다고 합시다. 처짐 문제의 잔차 $u''_\theta(x)-1$을 계산할 때는 **위치 $x$**에 대해 두 번 미분합니다. `tanh`처럼 매끄러운 활성화 함수를 쓰는 이유입니다.

반면 `loss.backward()`는 손실을 **신경망의 가중치 $\theta$**에 대해 미분합니다. L-BFGS는 `optimizer.step(closure)` 안에서 손실과 기울기를 다시 계산하며 가중치를 갱신합니다.


[양끝이 고정된 줄의 처짐 구하기](01_PINN.ipynb#lab-11) · [PINN 기초 추가 실습](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb#first-problem)


--- 

Further resources: [Open Hackathons](https://www.openhackathons.org/s/technical-resources) and the [OpenACC and Hackathons Slack channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.